# Importações

In [2]:
import pandas as pd
import requests

# Testes iniciais (conhecendo a API)

In [3]:
# Localização (São Paulo) - Teste inicial
latitude = -23.5505
longitude = -46.6333

# Intervalo de datas
start_date = "20240101"
end_date = "20241231"

# Variáveis climáticas
parameters = [
    "PRECTOTCORR",        # Precipitação
    "T2M",                # Temperatura média
    "RH2M",               # Umidade relativa
    "WS2M",               # Velocidade do vento
    "ALLSKY_SFC_SW_DWN"   # Radiação solar
]

parameters_str = ",".join(parameters)

In [4]:
url = (
    "https://power.larc.nasa.gov/api/temporal/daily/point"
    f"?parameters={parameters_str}"
    "&community=AG"
    f"&longitude={longitude}"
    f"&latitude={latitude}"
    f"&start={start_date}"
    f"&end={end_date}"
    "&format=JSON"
)

print("URL utilizada:")
print(url)

response = requests.get(url)

print("\nStatus da requisição:")
print(response.status_code)

URL utilizada:
https://power.larc.nasa.gov/api/temporal/daily/point?parameters=PRECTOTCORR,T2M,RH2M,WS2M,ALLSKY_SFC_SW_DWN&community=AG&longitude=-46.6333&latitude=-23.5505&start=20240101&end=20241231&format=JSON

Status da requisição:
200


In [5]:
# Transformando em Dataframe para consulta

data = response.json()

parameters_data = data["properties"]["parameter"]

df = pd.DataFrame(parameters_data)

df.index = pd.to_datetime(df.index)

df = df.reset_index()

df.rename(columns={"index": "date"}, inplace=True)

df.head()

,date,PRECTOTCORR,T2M,RH2M,WS2M,ALLSKY_SFC_SW_DWN
0,2024-01-01,0.06,23.15,69.16,2.99,16.71
1,2024-01-02,0.37,23.99,69.60,2.16,20.61
2,2024-01-03,4.72,24.20,75.18,2.47,16.73
3,2024-01-04,4.64,23.63,77.13,3.15,16.95
4,2024-01-05,2.36,23.16,75.31,2.68,21.71


In [6]:
# Validando os valores
print("Informações gerais:")
display(df.info())

print("\nValores nulos:")
display(df.isnull().sum())

print("\nEstatísticas:")
display(df.describe())

Informações gerais:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 366 entries, 0 to 365
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   date               366 non-null    datetime64[ns]
 1   PRECTOTCORR        366 non-null    float64       
 2   T2M                366 non-null    float64       
 3   RH2M               366 non-null    float64       
 4   WS2M               366 non-null    float64       
 5   ALLSKY_SFC_SW_DWN  366 non-null    float64       
dtypes: datetime64[ns](1), float64(5)
memory usage: 17.3 KB


None


Valores nulos:


,0
date,0
PRECTOTCORR,0
T2M,0
RH2M,0
WS2M,0
ALLSKY_SFC_SW_DWN,0



Estatísticas:


,date,PRECTOTCORR,T2M,RH2M,WS2M,ALLSKY_SFC_SW_DWN
count,366,366.000000,366.000000,366.000000,366.000000,366.000000
mean,2024-07-01 12:00:00.000000256,2.736120,21.516639,71.147131,2.298306,16.328251
min,2024-01-01 00:00:00,0.000000,10.230000,28.640000,0.830000,1.670000
25%,2024-04-01 06:00:00,0.030000,19.422500,63.820000,1.610000,12.962500
50%,2024-07-01 12:00:00,0.275000,22.190000,73.950000,2.140000,16.625000
75%,2024-09-30 18:00:00,3.067500,23.812500,81.057500,2.800000,20.305000
max,2024-12-31 00:00:00,39.170000,28.050000,93.140000,4.990000,29.240000
std,NaN,5.395045,3.259038,12.620045,0.840585,5.485059


In [7]:
df.to_csv(
    "nasa_power_sao_paulo_2024.csv",
    index=False
)

# Código para um protótipo inicial da ideia

In [8]:
#   Como fiz o teste para conhecer a API e só considerar SP, agora é acrescentar
# mais cidades do estado do Brasil para se aproximar de um protótipo da ideia final.
# Considerando 6 capitais agora

cities = {
    "Sao_Paulo": (-23.5505, -46.6333),
    "Rio_de_Janeiro": (-22.9068, -43.1729),
    "Curitiba": (-25.4284, -49.2733),
    "Florianopolis": (-27.5949, -48.5482),
    "Porto_Alegre": (-30.0346, -51.2177),
    "Belo_Horizonte": (-19.9167, -43.9345)
}

In [11]:
start_date = "20240101"
end_date = "20241231"

parameters = [
    "PRECTOTCORR",
    "T2M",
    "RH2M",
    "WS2M",
    "ALLSKY_SFC_SW_DWN"
]

parameters_str = ",".join(parameters)

In [12]:
def get_nasa_data(city, latitude, longitude):

    url = (
        "https://power.larc.nasa.gov/api/temporal/daily/point"
        f"?parameters={parameters_str}"
        "&community=AG"
        f"&longitude={longitude}"
        f"&latitude={latitude}"
        f"&start={start_date}"
        f"&end={end_date}"
        "&format=JSON"
    )

    response = requests.get(url)

    if response.status_code != 200:
        print(f"Erro ao consultar {city}")
        return None

    data = response.json()

    parameters_data = data["properties"]["parameter"]

    df = pd.DataFrame(parameters_data)

    df.index = pd.to_datetime(df.index)

    df = df.reset_index()

    df.rename(columns={"index": "date"}, inplace=True)

    df["city"] = city
    df["latitude"] = latitude
    df["longitude"] = longitude

    return df

In [13]:
all_dataframes = []

for city, coordinates in cities.items():

    latitude = coordinates[0]
    longitude = coordinates[1]

    print(f"Baixando dados de {city}...")

    city_df = get_nasa_data(
        city,
        latitude,
        longitude
    )

    if city_df is not None:
        all_dataframes.append(city_df)

Baixando dados de Sao_Paulo...
Baixando dados de Rio_de_Janeiro...
Baixando dados de Curitiba...
Baixando dados de Florianopolis...
Baixando dados de Porto_Alegre...
Baixando dados de Belo_Horizonte...


In [14]:
df = pd.concat(
    all_dataframes,
    ignore_index=True
)

In [15]:
print("Quantidade de linhas:")
print(len(df))

print("\nQuantidade de colunas:")
print(len(df.columns))

df.head()

Quantidade de linhas:
2196

Quantidade de colunas:
9


,date,PRECTOTCORR,T2M,RH2M,WS2M,ALLSKY_SFC_SW_DWN,city,latitude,longitude
0,2024-01-01,0.06,23.15,69.16,2.99,16.71,Sao_Paulo,-23.5505,-46.6333
1,2024-01-02,0.37,23.99,69.60,2.16,20.61,Sao_Paulo,-23.5505,-46.6333
2,2024-01-03,4.72,24.20,75.18,2.47,16.73,Sao_Paulo,-23.5505,-46.6333
3,2024-01-04,4.64,23.63,77.13,3.15,16.95,Sao_Paulo,-23.5505,-46.6333
4,2024-01-05,2.36,23.16,75.31,2.68,21.71,Sao_Paulo,-23.5505,-46.6333


## Validação dos dados

In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2196 entries, 0 to 2195
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   date               2196 non-null   datetime64[ns]
 1   PRECTOTCORR        2196 non-null   float64       
 2   T2M                2196 non-null   float64       
 3   RH2M               2196 non-null   float64       
 4   WS2M               2196 non-null   float64       
 5   ALLSKY_SFC_SW_DWN  2196 non-null   float64       
 6   city               2196 non-null   object        
 7   latitude           2196 non-null   float64       
 8   longitude          2196 non-null   float64       
dtypes: datetime64[ns](1), float64(7), object(1)
memory usage: 154.5+ KB


In [19]:
df.isnull().sum()

,0
date,0
PRECTOTCORR,0
T2M,0
RH2M,0
WS2M,0
ALLSKY_SFC_SW_DWN,0
city,0
latitude,0
longitude,0


In [20]:
df.describe()

,date,PRECTOTCORR,T2M,RH2M,WS2M,ALLSKY_SFC_SW_DWN,latitude,longitude
count,2196,2196.000000,2196.000000,2196.000000,2196.000000,2196.000000,2196.000000,2196.000000
mean,2024-07-01 12:00:00,3.512805,21.359212,76.627992,2.281940,16.089731,-24.905317,-47.129983
min,2024-01-01 00:00:00,0.000000,5.270000,23.860000,0.370000,0.710000,-30.034600,-51.217700
25%,2024-04-01 00:00:00,0.040000,19.470000,71.377500,1.480000,11.842500,-27.594900,-49.273300
50%,2024-07-01 12:00:00,0.425000,22.050000,79.415000,2.140000,16.310000,-24.489450,-47.590750
75%,2024-10-01 00:00:00,3.612500,24.000000,84.730000,2.922500,20.680000,-22.906800,-43.934500
max,2024-12-31 00:00:00,85.450000,29.270000,97.150000,7.220000,32.930000,-19.916700,-43.172900
std,NaN,7.468653,3.867850,11.765930,1.098421,6.478409,3.280303,2.870929


## Salvando o CSV: dados brutos

In [21]:
df.to_csv(
    "sentinel_sos_raw.csv",
    index=False
)

print("CSV salvo com sucesso!")

CSV salvo com sucesso!


## Engenharia de atributos


### Estação do ano

In [22]:
df["month"] = df["date"].dt.month

df["year"] = df["date"].dt.year

df["day"] = df["date"].dt.day

In [23]:
def get_season(month):

    if month in [12, 1, 2]:
        return "Summer"

    elif month in [3, 4, 5]:
        return "Autumn"

    elif month in [6, 7, 8]:
        return "Winter"

    else:
        return "Spring"

In [24]:
df["season"] = df["month"].apply(get_season)

### Índice de chuva

In [25]:
df["rain_intensity"] = (
    df["PRECTOTCORR"] *
    df["RH2M"]
)

### Índice de tempestade

In [26]:
df["storm_index"] = (
    df["PRECTOTCORR"] *
    df["WS2M"]
)

### Índice de calor

In [27]:
df["heat_index"] = (
    df["T2M"] *
    (df["RH2M"] / 100)
)

### Chuva extrema

In [28]:
df["extreme_rain"] = (
    df["PRECTOTCORR"] > 40
).astype(int)

### Criar variável alvo (risk_level)

In [29]:
def classify_risk(row):

    if row["PRECTOTCORR"] >= 50:
        return 2

    elif row["PRECTOTCORR"] >= 20:
        return 1

    return 0

In [30]:
df["risk_level"] = df.apply(
    classify_risk,
    axis=1
)

In [31]:
# Resultado df final
df.head()

,date,PRECTOTCORR,T2M,RH2M,WS2M,ALLSKY_SFC_SW_DWN,city,latitude,longitude,month,year,day,season,rain_intensity,storm_index,heat_index,extreme_rain,risk_level
0,2024-01-01,0.06,23.15,69.16,2.99,16.71,Sao_Paulo,-23.5505,-46.6333,1,2024,1,Summer,4.1496,0.1794,16.010540,0,0
1,2024-01-02,0.37,23.99,69.60,2.16,20.61,Sao_Paulo,-23.5505,-46.6333,1,2024,2,Summer,25.7520,0.7992,16.697040,0,0
2,2024-01-03,4.72,24.20,75.18,2.47,16.73,Sao_Paulo,-23.5505,-46.6333,1,2024,3,Summer,354.8496,11.6584,18.193560,0,0
3,2024-01-04,4.64,23.63,77.13,3.15,16.95,Sao_Paulo,-23.5505,-46.6333,1,2024,4,Summer,357.8832,14.6160,18.225819,0,0
4,2024-01-05,2.36,23.16,75.31,2.68,21.71,Sao_Paulo,-23.5505,-46.6333,1,2024,5,Summer,177.7316,6.3248,17.441796,0,0


### Salvando dataset final

In [32]:
df.to_csv(
    "sentinel_sos_features.csv",
    index=False
)

print("Dataset final salvo!")

Dataset final salvo!
